# Practical 11 – SVM, Random Forest, AdaBoost and GridSearchCV Combined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay


In [ ]:
raw = load_breast_cancer(as_frame=True)
X = raw.data
y = raw.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores = {'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred, pos_label=0), 'Recall': recall_score(y_test, pred, pos_label=0), 'F1': f1_score(y_test, pred, pos_label=0)}
    print(name, scores)
    print(classification_report(y_test, pred, target_names=['Malignant', 'Benign']))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0, 1], display_labels=['Malignant', 'Benign'])
    plt.title(name)
    plt.show()
    return scores


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
models = {'SVM': (Pipeline([('scale', StandardScaler()), ('clf', SVC())]), {'clf__C': [0.1, 1, 10], 'clf__kernel': ['linear', 'rbf']}), 'Random Forest': (RandomForestClassifier(random_state=42, n_jobs=-1), {'n_estimators': [50, 100], 'max_depth': [3, 5, None], 'min_samples_split': [2, 5]}), 'AdaBoost': (AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), random_state=42), {'n_estimators': [50, 100], 'learning_rate': [0.1, 0.5, 1.0]})}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
rows = []
for name, (model, grid) in models.items():
    search = GridSearchCV(model, grid, scoring=make_scorer(f1_score, pos_label=0), cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    print(name, 'Best parameters:', search.best_params_, 'CV F1:', search.best_score_)
    scores = {'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred, pos_label=0), 'Recall': recall_score(y_test, pred, pos_label=0), 'F1': f1_score(y_test, pred, pos_label=0)}
    print('Test scores:', scores)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0, 1], display_labels=['Malignant', 'Benign'])
    plt.title(name)
    plt.show()
    rows.append({'Algorithm': name, **scores, 'CV F1': search.best_score_})


In [ ]:
result = pd.DataFrame(rows)
print(result.round(4))
result.set_index('Algorithm')[['Accuracy', 'Precision', 'Recall', 'F1']].plot.bar(rot=0, ylim=(0, 1.05), title='Model Comparison')
plt.tight_layout()
plt.show()
result.set_index('Algorithm')['CV F1'].plot.bar(rot=0, ylim=(0, 1.05), title='Cross Validation F1')
plt.show()


In [ ]:
result.to_csv('pr11_model_comparison.csv', index=False)
